# PSI Structural Damage Assessor
**Photo Severity Index** — adaptive thresholding + contour analysis pipeline.

This notebook explores the PSI algorithm on sample images.
All logic lives in `../core/psi.py`; this notebook is for experimentation and visualization.


In [ ]:
import sys
from pathlib import Path

# make core/ importable from notebook/
sys.path.insert(0, str(Path("..")))  

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from PIL import Image

from core.psi import calculate_psi, SEVERITY_COLORS, SEVERITY_RUBRIC


## Load image


In [ ]:
IMAGE_PATH = Path("../assets/original.jpg")
assert IMAGE_PATH.exists(), f"Not found: {IMAGE_PATH}"

image_rgb = np.array(Image.open(IMAGE_PATH).convert("RGB"))
print(f"Loaded: {IMAGE_PATH.name}  shape={image_rgb.shape}  dtype={image_rgb.dtype}")


## Run PSI pipeline


In [ ]:
result = calculate_psi(image_rgb)

color = SEVERITY_COLORS[result.psi_index]
print(f"Damage  : {result.damage_pct:.2f}%")
print(f"PSI     : {result.psi_index}")
print(f"Category: {result.category}")
print(f"Runtime : {result.runtime:.3f}s")


## Visualize results


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor("#090c12")

panels = [
    (result.original, "Original"),
    (result.roi,      "ROI Bounding Boxes"),
    (result.masked,   "Damage Mask"),
]

for ax, (img, title) in zip(axes, panels):
    ax.imshow(img)
    ax.set_title(title, color="#e2e8f0", fontsize=11, pad=8)
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor("#1e2d42")

fig.suptitle(
    f"PSI {result.psi_index} — {result.category}  ({result.damage_pct:.2f}% damage)",
    color=color, fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()


## Severity rubric


In [ ]:
col1, col2, col3 = "Max %", "PSI", "Category"
print(f"{col1:<8} {col2:<6} {col3}")
print("-" * 36)
for upper, psi, category in SEVERITY_RUBRIC:
    upper_str = "<=" + (str(int(upper)) if upper != float("inf") else "100")
    marker = " <--" if psi == result.psi_index else ""
    print(f"{upper_str:<8} {psi:<6} {category}{marker}")
